# Module 8: Model Blueprint and Evaluation Plan

Continue the **Will They Show Up?** campus-events project. Module 8 plans and audits the model; Module 9 performs the complete build.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=e8ab9e1a-2c93-4e8c-aa67-b49c00fc87e6"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
import json
from pathlib import Path
import pandas as pd

try:
    from sklearn.model_selection import train_test_split
except ModuleNotFoundError:
    train_test_split = None

df = pd.read_csv('campus_events_clean.csv')
dictionary = pd.read_csv('campus_events_data_dictionary.csv')
print('Dataset:', df.shape)
df.head()

## 1. Intake the Module 7 handoff

Use your own `m7_feature_candidates_for_m8.csv`. If it is unavailable, use the supplied checkpoint and still document what you would verify.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=e92466d3-2b21-4836-8312-b49c00fc8835"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
student_handoff = Path('m7_feature_candidates_for_m8.csv')
checkpoint = Path('m7_feature_candidates_for_m8_checkpoint.csv')
feature_path = student_handoff if student_handoff.exists() else checkpoint
feature_review = pd.read_csv(feature_path)
print('Using:', feature_path.name)
feature_review

## 2. Define the prediction job

The classroom task is classification: predict whether a fictional event will have **high turnout**, using only information reasonably available before the event. This is planning support, not permission to make real funding, access, or disciplinary decisions.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=0573f005-bbc0-45a1-a6cc-b49c00fcd1ec"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
prediction_question = 'Can pre-event information predict whether a fictional campus event will have high turnout?'
target = 'high_turnout'
blocked_features = {
    'event_id': 'identifier',
    'actual_weather': 'not known at planning time',
    'actual_attendance': 'post-event outcome; direct leakage',
    'attendance_rate': 'directly determines the target',
    'high_turnout': 'target answer',
}
pd.DataFrame(blocked_features.items(), columns=['feature', 'rejection_reason'])

### Complete the feature audit

For every candidate, decide whether it is available before prediction, supported by M7 evidence, free of direct leakage, not an identifier, and acceptable after a proxy-risk review.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=0a05e43d-fb6d-43ad-b7d2-b49c00fccc64"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
required_columns = ['candidate_feature','m7_evidence','available_before_prediction','direct_leakage','identifier','proxy_risk','decision','reason']
for column in required_columns:
    if column not in feature_review.columns:
        feature_review[column] = ''
feature_review[required_columns]

## 3. Plan the split

Use a stratified split so both labels are represented. In a larger project, use train/validation/test or cross-validation during model selection and reserve the final test set. This M8 rehearsal uses train/test only and does not train or select the final model.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=1e88231b-4f7f-4383-a224-b49c00fce0a5"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
y = df[target].map({'yes': 1, 'no': 0})
if train_test_split is not None:
    train_index, test_index = train_test_split(df.index, test_size=0.25, random_state=42, stratify=y)
else:
    train_parts, test_parts = [], []
    for _, group in df.assign(_target=y).groupby('_target'):
        shuffled = group.sample(frac=1, random_state=42)
        n_test = max(1, round(len(shuffled) * 0.25))
        test_parts.extend(shuffled.index[:n_test])
        train_parts.extend(shuffled.index[n_test:])
    train_index, test_index = train_parts, test_parts
print('Training rows:', len(train_index), 'Testing rows:', len(test_index))

## 4. Establish the baseline

The baseline is the majority label in the training set. A model must be compared with this simple guess, but beating it does not by itself make the model useful or safe.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=e1639ef9-9d30-4ceb-925e-b49c00fd0424"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
y_train = y.loc[train_index]
y_test = y.loc[test_index]
baseline_label = int(y_train.mode().iloc[0])
baseline_accuracy = float((y_test == baseline_label).mean())
class_balance = y.value_counts(normalize=True).sort_index().round(3).to_dict()
print('Class balance:', class_balance)
print('Baseline label:', baseline_label, 'Baseline accuracy:', round(baseline_accuracy, 3))

## 5. Choose the metric and name the error costs

Accuracy is the primary classroom comparison because the synthetic labels are not extremely imbalanced. Also inspect recall for high-turnout events and the confusion matrix in M9. Decide which error matters more for the stated use—do not let the notebook decide that value judgment for you.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=99cbe365-f941-4a92-9b07-b49c00fd11f3"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
primary_metric = 'accuracy, interpreted with class balance and the confusion matrix'
false_positive_cost = 'Resources may be planned for turnout that does not occur.'
false_negative_cost = 'A potentially strong event may receive too little planning support.'
more_costly_error = 'Student decision: explain which cost matters more for the intended low-stakes planning use.'

## 6. Predict failure cases before building

Name situations where future events may differ from this synthetic dataset.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=983d2c8e-0ad1-4a9f-8ded-b49c00fd1f40"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
predicted_failure_cases = [
    'An unusual event type or venue not represented in the training data',
    'A sudden weather or campus-schedule change after the prediction is made',
    'Promotion practices change, so old relationships no longer transfer',
]
predicted_failure_cases

## 7. State the responsibility boundary

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=ccf0232f-c510-4a2d-9c76-b49c00fd4673"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
prohibited_uses = [
    'Do not use the model to deny students access to events.',
    'Do not use it as the sole basis for funding or staffing decisions.',
    'Do not apply this synthetic classroom model to real people or institutions.',
]
human_review_boundary = 'A person must review context, uncertainty, and consequences before any real planning decision.'

## 8. Export the M9 handoff

Review every field. Module 9 must open this blueprint before building and explain any revision.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="https://stevens.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=bcd79cb3-395d-4ae9-9f0c-b49c00fd4d9e"><img src="https://img.shields.io/badge/Watch_Video-0B7A53?style=flat&logo=youtubemusic&logoColor=white"></a></td></tr></table>

In [ ]:
accepted = feature_review.loc[feature_review['decision'].astype(str).str.lower().eq('accept'), 'candidate_feature'].tolist()
rejected = feature_review.loc[~feature_review['decision'].astype(str).str.lower().eq('accept'), ['candidate_feature','reason']].fillna('').to_dict('records')
blueprint = {
    'project': 'Will They Show Up?',
    'prediction_question': prediction_question,
    'target': target,
    'accepted_features': accepted,
    'rejected_features': rejected,
    'split_plan': 'M8 rehearsal: stratified 75/25 train/test; M9 model selection must not use the final test set',
    'class_balance': class_balance,
    'majority_baseline_accuracy': round(baseline_accuracy, 3),
    'primary_metric': primary_metric,
    'false_positive_cost': false_positive_cost,
    'false_negative_cost': false_negative_cost,
    'more_costly_error': more_costly_error,
    'predicted_failure_cases': predicted_failure_cases,
    'prohibited_uses': prohibited_uses,
    'human_review_boundary': human_review_boundary,
}
Path('m8_model_blueprint.json').write_text(json.dumps(blueprint, indent=2))
feature_review.to_csv('m8_reviewed_feature_table.csv', index=False)
print('Saved m8_model_blueprint.json and m8_reviewed_feature_table.csv')
blueprint